In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import defaultdict
import math
from matplotlib.patches import Ellipse
from skimage.measure import EllipseModel
import matplotlib.animation as animation
import matplotlib as mpl
from IPython.display import HTML
from scipy.signal import butter, filtfilt
import json

mpl.rcParams['animation.html'] = 'jshtml'
mpl.rcParams['animation.embed_limit'] = 2**128

In [ ]:
def readEllipse(file_path):
    '''
    Reads the file to recreate the ellipses
    '''

    with open(file_path, 'r') as f:
        lines = f.readlines()

    ellipses = []

    for line in lines:
        parts = list(map(float, line.strip().split()))

        class_id = int(parts[0])
        points = np.array(parts[1:]).reshape(-1, 2)
        center = np.mean(points, axis=0)
        centered_points = points - center
        cov = np.cov(centered_points.T)
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        major_axis = 2 * math.sqrt(max(eigenvalues))
        minor_axis = 2 * math.sqrt(min(eigenvalues))
        angle = math.degrees(math.atan2(eigenvectors[1, 0], eigenvectors[0, 0]))

        ellipses.append({
            'class_id': class_id,
            'center': center,
            'major_axis': major_axis,
            'minor_axis': minor_axis,
            'angle': angle,
            'points': points
        })

    return ellipses

In [ ]:
def fitEllipse(points, screen_res):
    '''
    Recreates the ellipse to output parameters and angle to the camera plane
    '''

    ell = EllipseModel()
    ell.estimate(points)
    xc, yc, a, b, theta = ell.params

    xc = xc*screen_res[0]
    yc = screen_res[1]*(1-yc)
    a = a*screen_res[1]
    b = b*screen_res[0]

    if b < a:
        angle2camera = np.arccos(b/a)
    else:
        angle2camera = 0

    ellipse_data = (xc, yc, a, b, theta, angle2camera)

    # Unit test -----
    b_test = 10
    a_test = 10
    assert np.arccos(b_test/a_test) == 0, 'Arccos(1) != 0'
    # ----- ----- -----

    return ellipse_data

In [ ]:
def process_directory(directory_path, screen_resolution):
    '''
    Process directory to extract the data from .txt files
    '''


    files = [f for f in os.listdir(directory_path) if f.endswith('.txt')]
    files.sort()#key=natural_sort_key)

    comodin = files[0].split('_')
    first_frame = int((comodin[1].split('.'))[0])
    
    tracking_data = defaultdict(lambda: {
        'x_pos': [], 'y_pos': [], 'major_axes': [], 'minor_axes': [],
        'frames': [], 'normal_x_pos': [], 'normal_y_pos': [], 'points': [], 
        'ellipses': []
    })
    
    for frame_idx, filename in enumerate(files):
        file_path = os.path.join(directory_path, filename)
        for ellipse in readEllipse(file_path):
            class_id = ellipse['class_id']
            center_x = screen_resolution[0] * ellipse['center'][0]
            center_y = screen_resolution[1] * (1 - ellipse['center'][1])
            tracking_data[class_id]['x_pos'].append(center_x)
            tracking_data[class_id]['y_pos'].append(center_y)
            tracking_data[class_id]['normal_x_pos'].append(ellipse['center'][0])
            tracking_data[class_id]['normal_y_pos'].append(1 - ellipse['center'][1])
            tracking_data[class_id]['major_axes'].append(ellipse['major_axis'])
            tracking_data[class_id]['minor_axes'].append(ellipse['minor_axis'])
            tracking_data[class_id]['points'].append(ellipse['points'])
            # tracking_data[class_id]['angle'] = ellipse['angle']
            tracking_data[class_id]['frames'].append(frame_idx)
            tracking_data[class_id]['ellipses'].append(fitEllipse(ellipse['points'], screen_resolution))
    
    # tracking_data['length'] = len(files)
    tracking_data['first_frame'] = first_frame
    return tracking_data

In [ ]:
def findExtremes(points, screen_res):
    '''
    Find the 4 extreme points of an ellipse
    '''


    points = np.array(points)
    x_coords = points[:, 0] * screen_res[0]
    y_coords = (1-points[:, 1]) * screen_res[1]

    # The -1 in y_coords is due to the change of axis (y = 1 - y)

    min_x = np.min(x_coords); idx_min_x = np.argmin(x_coords)
    max_x = np.max(x_coords); idx_max_x = np.argmax(x_coords)
    min_y = np.min(y_coords); idx_min_y = np.argmin(y_coords)
    max_y = np.max(y_coords); idx_max_y = np.argmax(y_coords)


    bottom_coord = np.array([x_coords[idx_min_y], min_y])
    top_coord = np.array([x_coords[idx_max_y], max_y])
    left_coord = np.array([min_x, y_coords[idx_min_x]])
    right_coord = np.array([max_x, y_coords[idx_max_x]])
    
    vertical_dist = np.sqrt((top_coord[0]-bottom_coord[0])**2 + (top_coord[1]-bottom_coord[1])**2)
    horizontal_dist = np.sqrt((right_coord[0]-left_coord[0])**2 + (right_coord[1]-left_coord[1])**2)

    distances = np.array([vertical_dist, horizontal_dist])

    return bottom_coord, top_coord, left_coord, right_coord, distances

In [ ]:
def isReliable(point, screen_resolution, reliable_area):
    '''
    Check if the ellipse is within the reliable area to avoid distortion
    '''

    x, z = point


    return

In [ ]:
def ellipse2angle(ellipse):
    '''
    Create a single vector containing the angles to the camera for plotting
    '''
    angles = []
    
    for i in range(len(ellipse)):
        angles.append(ellipse[i][-1])

    return angles

In [ ]:
def butter_lowpass(cutoff, fs, order=5):
    '''
    Design a Butterworth lowpass filter 
    ''' 
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a


def lowpass_filter(data, cutoff=2.0, fs=30.0, order=5):
    '''
    Apply lowpass filter to data
    '''

    b, a = butter_lowpass(cutoff, fs, order=order)
    y = filtfilt(b, a, data)
    return y

In [ ]:
def apply_filters(tracking_data, cutoff=2.0, fs=30.0):
    '''
    Apply lowpass filters to all tracking data
    '''
    
    filtered_data = {}
    for class_id, data in tracking_data.items():
        if class_id == 0 or class_id == 1:
            filtered = {
                'x_pos': lowpass_filter(data['x_pos'], cutoff, fs),
                'y_pos': lowpass_filter(data['y_pos'], cutoff, fs),
                'major_axes': lowpass_filter(data['major_axes'], cutoff, fs),
                'minor_axes': lowpass_filter(data['minor_axes'], cutoff, fs),
                'frames': data['frames']
            }
            filtered_data[class_id] = filtered

    return filtered_data

In [ ]:
def save2csv(frames, slopes, name):
    '''
    Saves the data in a csv file
    '''

    directory = 'data/slope/'
    np.savetxt(directory+name, [p for p in zip(frames, slopes)], delimiter=',', fmt='%s')

    return

In [ ]:
def find_calibration(tracking_data, screen_resolution):
    '''
    Finds the calibration frame assuming the bicycle only goes near the center once.
    It looks for the frame where both wheels are in the closest point to the center of the screen.
    '''

    x_diff2center_f = np.array(tracking_data[0]['x_pos']) - 0.5 * screen_resolution[0]
    x_diff2center_r = np.array(tracking_data[1]['x_pos']) - 0.5 * screen_resolution[0]

    y_diff2center_f = np.array(tracking_data[0]['y_pos']) - 0.5 * screen_resolution[1]
    y_diff2center_r = np.array(tracking_data[1]['y_pos']) - 0.5 * screen_resolution[1]

    centered_f = np.sqrt(x_diff2center_f**2 + y_diff2center_f**2)
    centered_r = np.sqrt(x_diff2center_r**2 + y_diff2center_r**2)

    sum_diff = centered_f + centered_r
    idx_min = np.argmin(sum_diff)

    calibration_frame = idx_min+1

    return calibration_frame

In [ ]:
def calc_unit_vectors(ellipse_front, ellipse_rear):
    '''
    Calculate the unit vectors attached to the ellipses
    '''

    # Revise this code and change the axes to be x-z

    xf, yf = ellipse_front['center']
    xr, yr = ellipse_rear['center']

    a_f = ellipse_front['major_axis']
    b_f = ellipse_front['minor_axis']

    a_r = ellipse_rear['major_axis']
    b_r = ellipse_rear['minor_axis']

    theta_f = ellipse_front['angle']
    theta_r = ellipse_rear['angle']

    if xf <= xr:
        x1f, y1f = [xf, xf + b_f*np.cos(theta_f)], [yf, yf + b_f*np.sin(theta_f)]
        x2f, y2f = [xf, xf - a_f*np.sin(theta_f)], [yf, yf + a_f*np.cos(theta_f)]

        x1r, y1r = [xr, xr + b_r*np.cos(theta_r)], [yr, yr + b_r*np.sin(theta_r)]
        x2r, y2r = [xr, xr - a_r*np.sin(theta_r)], [yr, yr + a_r*np.cos(theta_r)]
    else:
        x1f, y1f = [xf, xf + b_f*np.cos(theta_f)], [yf, yf + b_f*np.sin(theta_f)]
        x2f, y2f = [xf, xf + a_f*np.sin(theta_f)], [yf, yf + a_f*np.cos(theta_f)]

        x1r, y1r = [xr, xr + b_r*np.cos(theta_r)], [yr, yr + b_r*np.sin(theta_r)]
        x2r, y2r = [xr, xr + a_r*np.sin(theta_r)], [yr, yr + a_r*np.cos(theta_r)]

    horizontal_f = [x2f, y2f]
    vertical_f = [x1f, y1f]

    horizontal_r = [x2r, y2r]
    vertical_r = [x1r, y1r]

    vectors_f = [horizontal_f, vertical_f]
    vectors_r = [horizontal_r, vertical_r]

    return vectors_f, vectors_r

In [ ]:
def clean4json(obj):
    # numpy arrays -> lists
    if isinstance(obj, np.ndarray):
        return [clean4json(x) for x in obj.tolist()]
    # numpy scalars -> python scalars
    if isinstance(obj, (np.floating, np.integer, np.bool_)):
        return obj.item()
    # handle floats: replace NaN/Inf with None (strict JSON)
    if isinstance(obj, float):
        if not math.isfinite(obj):
            # return None
            return 0
        return obj
    # containers
    if isinstance(obj, dict):
        return {k: clean4json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [clean4json(x) for x in obj]
    # pass through other JSON-safe types (str, int, bool, None)
    return obj


In [ ]:
def data4model(track_data, assumed_origin):
    '''
    Create the nested dictionary by frame for the data required in the model of the bicycle
    '''

    data = {
        'Ry_x':0,
        'Ry_z':0,
        'Fy_x':0,
        'Fy_z':0,
        'r_Cf_Cr_x':0,
        'r_Cf_Cr_z':0,
        'r_Cr_O_x':0,
        'r_Cr_O_z':0,
        'r_P_S_x':0,
        'r_P_S_z':0
    }

    rear_wheel = {
        'x': 0,
        'z': 0,
        'psi': 0
    }

    ellipse_r = np.array(track_data[1]['ellipses'])
    ellipse_f = np.array(track_data[0]['ellipses'])

    x_r = np.array(ellipse_r[:,0])
    z_r = np.array(ellipse_r[:,1])
    theta_r = ellipse_r[:,4]
    angle_cam_r = ellipse_r[:,5]

    ellipse_r_data = [x_r, z_r, theta_r, angle_cam_r]

    x_f = np.array(ellipse_f[:,0])
    z_f = np.array(ellipse_f[:,1])
    theta_f = ellipse_f[:,4]
    angle_cam_f = ellipse_f[:,5]

    ellipse_f_data = [x_f, z_f, theta_f, angle_cam_f]

    ellipse_data = [ellipse_r_data, ellipse_f_data]

    rear_wheel['x'] = x_r
    rear_wheel['z'] = z_r
    rear_wheel['psi'] = angle_cam_r

    data['Ry_x'] = np.sin(angle_cam_r)*np.cos(theta_r)
    data['Ry_z'] = np.sin(angle_cam_r)*np.sin(theta_r)

    data['Fy_x'] = np.sin(angle_cam_f)*np.cos(theta_f)
    data['Fy_z'] = np.sin(angle_cam_f)*np.sin(theta_f)

    data['r_Cf_Cr_x'] = x_f - x_r
    data['r_Cf_Cr_z'] = z_f - z_r
    data['r_Cr_O_x'] = x_r - assumed_origin[0]
    data['r_Cr_O_z'] = z_r - assumed_origin[1]
    data['r_P_S_x'] = (np.sin(angle_cam_f)*np.cos(angle_cam_r) - np.sin(angle_cam_r)*np.cos(angle_cam_f)*np.cos(theta_f - theta_r))*np.sin(theta_f)/np.sqrt((np.sin(angle_cam_f)*np.cos(angle_cam_r) - np.sin(angle_cam_r)*np.cos(angle_cam_f)*np.cos(theta_f - theta_r))**2 + np.sin(angle_cam_r)**2*np.sin(theta_f - theta_r)**2) + np.sin(angle_cam_r)*np.sin(theta_f - theta_r)*np.cos(angle_cam_f)*np.cos(theta_f)/np.sqrt((np.sin(angle_cam_f)*np.cos(angle_cam_r) - np.sin(angle_cam_r)*np.cos(angle_cam_f)*np.cos(theta_f - theta_r))**2 + np.sin(angle_cam_r)**2*np.sin(theta_f - theta_r)**2)
    data['r_P_S_z'] = (np.sin(angle_cam_f)*np.cos(angle_cam_r) - np.sin(angle_cam_r)*np.cos(angle_cam_f)*np.cos(theta_f - theta_r))*np.cos(theta_f)/np.sqrt((np.sin(angle_cam_f)*np.cos(angle_cam_r) - np.sin(angle_cam_r)*np.cos(angle_cam_f)*np.cos(theta_f - theta_r))**2 + np.sin(angle_cam_r)**2*np.sin(theta_f - theta_r)**2) - np.sin(angle_cam_r)*np.sin(theta_f)*np.sin(theta_f - theta_r)*np.cos(angle_cam_f)/np.sqrt((np.sin(angle_cam_f)*np.cos(angle_cam_r) - np.sin(angle_cam_r)*np.cos(angle_cam_f)*np.cos(theta_f - theta_r))**2 + np.sin(angle_cam_r)**2*np.sin(theta_f - theta_r)**2)

    data_json = clean4json(data)
    rw_json = clean4json(rear_wheel)

    

    return data_json, rw_json

In [ ]:
# Settings

video_id = 'gopro'

if len(video_id) <= 3:
    directory_path_min = f'/home/eimolgon/Documents/PhD-Project/02-video-data/yt-crash-{video_id}/labels/train'
    directory_path_may = f'/home/eimolgon/Documents/PhD-Project/02-video-data/yt-crash-{video_id}/labels/Train'
else:
    # directory_path_min = f'/home/eimolgon/Documents/PhD-Project/02-video-data/gopro_test_1_1_resize_annotation_161025/labels/train'
    # directory_path_may = f'/home/eimolgon/Documents/PhD-Project/02-video-data/gopro_test_1_1_resize_annotation_161025/labels/Train'
    directory_path_min = f'/home/eimolgon/Documents/PhD-Project/02-video-data/gopro_test_1_1_cut/labels/train'
    directory_path_may = f'/home/eimolgon/Documents/PhD-Project/02-video-data/gopro_test_1_1_cut/labels/Train'



# Set assumed screen resolution to go back from normalized data
screen_resolution = (1920, 1080)
reliable_area = (0.2, 0.8) # Assuming that this portion of the screen is less affected by barrel distortion
diameter_wheel = 0.6604 # For a 28 inch wheel
true_wheelbase = 1.02
imposed_origin = (reliable_area[0]* screen_resolution[0], reliable_area[0]* screen_resolution[1]) # This is only valid for videos where the bicycle moves in the xy plane and no depth is needed

reliable_x = [reliable_area[0] * screen_resolution[0], reliable_area[1] * screen_resolution[0]]
reliable_y = [reliable_area[0] * screen_resolution[1], reliable_area[1] * screen_resolution[1]]

# Process data
try:
    tracking_data = process_directory(directory_path_min, screen_resolution)
    
except FileNotFoundError:
    tracking_data = process_directory(directory_path_may, screen_resolution)

filtered_data = apply_filters(tracking_data)
data_model, rear_wheel_reconstruct = data4model(tracking_data, imposed_origin)

# write data to txt to import it on the model
# with open(f'data4model_{video_id}.json', 'w') as f:
#     json.dump(data_model, f)

with open(f'/home/eimolgon/Documents/PhD-Project/vid2dyn/output/data4model_{video_id}.json', 'w') as f:
    json.dump(data_model, f)


# with open(f'data_rear_wheel_{video_id}.json', 'w') as f:
    # json.dump(rear_wheel_reconstruct, f)


# Frame regularization for videos where wheels are not detected on the first frame
frame_count = len(tracking_data[0]['ellipses'])
first_frame = tracking_data['first_frame']
actual_frames = np.array(tracking_data[0]['frames']) + first_frame




x_diff = np.array(tracking_data[0]['x_pos']) - np.array(tracking_data[1]['x_pos'])
y_diff = np.array(tracking_data[0]['y_pos']) - np.array(tracking_data[1]['y_pos'])

wheelbase_screen = np.sqrt(x_diff**2 + y_diff**2)

# wb_screen2 = np.sqrt(x_diff2**2 + y_diff2**2)

slopes = (np.array(tracking_data[0]['y_pos']) - np.array(tracking_data[1]['y_pos'])) / (np.array(tracking_data[0]['x_pos']) - np.array(tracking_data[1]['x_pos']))


calibration_frame = find_calibration(tracking_data, screen_resolution)

points_f = np.array(tracking_data[0]['points'][calibration_frame])
points_r = np.array(tracking_data[1]['points'][calibration_frame])

# vertical = max(np.array(tracking_data[0]['points'][idx_min][0])) - min(np.array(tracking_data[0]['points'][idx_min][0]))

bottom_r, top_r, left_r, right_r, axes_r = findExtremes(points_r, screen_resolution)
bottom_f, top_f, left_f, right_f, axes_f = findExtremes(points_f, screen_resolution)


wheelbase_real = (diameter_wheel * wheelbase_screen) / axes_r[0]

angle_ellipse_f = ellipse2angle(tracking_data[0]['ellipses'])
angle_ellipse_r = ellipse2angle(tracking_data[1]['ellipses'])
steering_angle = np.array(angle_ellipse_f) - np.array(angle_ellipse_r)

grad_f = np.gradient(angle_ellipse_f)
grad_r = np.gradient(angle_ellipse_r)


actual_angles_f = []

for angle, gradient in zip(angle_ellipse_f, grad_f):
    if gradient <= 0:
        angle = 90 + (90 - angle)
        actual_angles_f.append(angle)



print('Frame used for calibration:', calibration_frame)
print('estimated wheelbase:', wheelbase_real[calibration_frame])
print('avg wheelbase:', np.mean(wheelbase_real))
print('error = ', (true_wheelbase - wheelbase_real[calibration_frame])/true_wheelbase * 100)


# These lines are used to check if the end point of the wheels are within the reliable area of the screen
print('x_f:', np.array(tracking_data[0]['x_pos'][-1]))
print('x_r:', np.array(tracking_data[1]['x_pos'][-1]))
print('y_f:', np.array(tracking_data[0]['y_pos'][-1]))
print('y_r:', np.array(tracking_data[1]['y_pos'][-1]))
print('reliable x:', reliable_x)
print('reliable y:', reliable_y)



# Uncomment to save the files
# namefile = f'slopes-ytcrash-{video_id}.csv'
# save2csv(actual_frames, slopes, namefile)


plt.plot(actual_frames, slopes)
plt.xlabel('Frame')
plt.ylabel('Slope')
plt.show()

# Change this for the reliable area of the screen
limit_i = int(frame_count*0.2)
limit_f = int(frame_count*0.8)

x_plot_reliable = np.linspace(limit_i, limit_f, limit_f-limit_i)

z = np.polyfit(x_plot_reliable, wheelbase_real[limit_i:limit_f], 1)
p = np.poly1d(z)

plt.plot(x_plot_reliable, wheelbase_real[limit_i:limit_f])
plt.plot(x_plot_reliable, p(x_plot_reliable), '--')
# plt.xlim(0,140)
plt.show()
    

plt.plot(actual_frames, np.rad2deg(angle_ellipse_f), label = 'Front wheel angle')
plt.plot(actual_frames, np.rad2deg(angle_ellipse_r), label = 'Rear wheel angle')
plt.plot(actual_frames, np.rad2deg(steering_angle), label = 'Steering angle')
plt.xlabel('Frame')
plt.ylabel('Angle (degrees)')
plt.legend()
plt.grid()
plt.show()


plt.plot(actual_frames, grad_f, label = 'Front wheel angle gradient')
plt.plot(actual_frames, grad_r, label = 'Rear wheel angle gradient')
plt.legend()
plt.grid()
plt.show()

# plt.plot(data_model[Rx_x])

In [ ]:
# 2D animation

# %matplotlib notebook
fig, ax = plt.subplots(figsize = (16,9))
ax.axis('equal')
# plt.gca().set_aspect('equal')


xf = np.asarray(tracking_data[0]['x_pos'])
yf = np.asarray(tracking_data[0]['y_pos'])

xr = np.asarray(tracking_data[1]['x_pos'])
yr = np.asarray(tracking_data[1]['y_pos'])

line2 = ax.plot(xf[0], yf[0], label=f'front wheel')[0]
dot2 = ax.plot(xf[0], yf[0], 'bo')[0]
line3 = ax.plot(xr[0], yr[0], label=f'rear wheel')[0]
dot3 = ax.plot(xr[0], yr[0], 'ro')[0]

ax.set(xlim=[0, screen_resolution[0]], ylim=[0, screen_resolution[1]], xlabel='X [pixels]', ylabel='Y [pixels]')


def update(frame):
    # for each frame, update the data stored on each artist.
    xf2 = xf[:frame]
    yf2 = yf[:frame]

    xr2 = xr[:frame]
    yr2 = yr[:frame]

    # update the line plot:
    line2.set_xdata(xf2[:frame])
    line2.set_ydata(yf2[:frame])
    dot2.set_data([xf[frame]], [yf[frame]])

    line3.set_xdata(xr2[:frame])
    line3.set_ydata(yr2[:frame])
    dot3.set_data([xr[frame]], [yr[frame]])

    
    ax.set_title(f'Steering angle: {round(np.rad2deg(steering_angle[frame]), 2)}, Frame: {first_frame + frame}, Time: {round(frame/30, 2)} s')
    
    return line2, line3


ani = animation.FuncAnimation(fig=fig, func=update, frames=len(xf), interval=60)
# plt.show()
# ani

HTML(ani.to_jshtml())

In [ ]:
# 2D animation filtered

# %matplotlib notebook
fig, ax = plt.subplots(figsize = (16,9))
ax.axis('equal')
# plt.gca().set_aspect('equal')


xf = np.asarray(filtered_data[0]['x_pos'])
yf = np.asarray(filtered_data[0]['y_pos'])

xr = np.asarray(filtered_data[1]['x_pos'])
yr = np.asarray(filtered_data[1]['y_pos'])

line2 = ax.plot(xf[0], yf[0], label=f'front wheel')[0]
dot2 = ax.plot(xf[0], yf[0], 'bo')[0]
line3 = ax.plot(xr[0], yr[0], label=f'rear wheel')[0]
dot3 = ax.plot(xr[0], yr[0], 'ro')[0]

ax.set(xlim=[0, screen_resolution[0]], ylim=[0, screen_resolution[1]], xlabel='X [pixels]', ylabel='Y [pixels]')

def update(frame):
    # for each frame, update the data stored on each artist.
    xf2 = xf[:frame]
    yf2 = yf[:frame]

    xr2 = xr[:frame]
    yr2 = yr[:frame]

    # update the line plot:
    line2.set_xdata(xf2[:frame])
    line2.set_ydata(yf2[:frame])
    dot2.set_data([xf[frame]], [yf[frame]])

    line3.set_xdata(xr2[:frame])
    line3.set_ydata(yr2[:frame])
    dot3.set_data([xr[frame]], [yr[frame]])

    ax.set_title(f'Steering angle: {round(np.rad2deg(steering_angle[frame]), 2)}, Frame: {first_frame + frame}, Time: {round(frame/30, 2)} s')
    
    return line2, line3


ani = animation.FuncAnimation(fig=fig, func=update, frames=len(xf), interval=60)
# plt.show()
# ani

HTML(ani.to_jshtml())

In [ ]:
# 3D animation

# Example setup
fig, ax = plt.subplots(figsize=(16, 9))
ax.axis('equal')

fwheel = np.array(tracking_data[0]['ellipses'])[:, 0:5]
rwheel = np.array(tracking_data[1]['ellipses'])[:, 0:5]

iota = np.deg2rad(25)


xf = np.asarray(fwheel[:,0])
yf = np.asarray(fwheel[:,1])
af = np.asarray(fwheel[:, 2])
bf = np.asarray(fwheel[:, 3])
thetaf = np.asarray(fwheel[:, 4])

xr = rwheel[:,0]
yr = rwheel[:,1]
ar = np.asarray(rwheel[:, 2])
br = np.asarray(rwheel[:, 3])
thetar = np.asarray(rwheel[:, 4])


# Create ellipse patches (initial position)
front_ellipse = Ellipse((xf[0], yf[0]), width=2*bf[0], height=2*af[0], 
                        angle=thetaf[0], edgecolor='blue',  facecolor='none')
rear_ellipse = Ellipse((xr[0], yr[0]), width=2*br[0], height=2*ar[0], 
                       angle=thetar[0], edgecolor='red', facecolor='none')

ax.add_patch(front_ellipse)
ax.add_patch(rear_ellipse)

# ax.plot() This is for the unitary vectors.

xs = xr + 2.5 * ar * np.cos(iota)
ys = yr + 2.5 * ar * np.sin(iota)

(line,) = ax.plot([xr[0], xf[0]], [yr[0], yf[0]], 'k-', lw=2)

# (line_rs,) = ax.plot([xr[0], xs[0]], [yr[0], ys[0]], 'g--', lw=2) 
# (line_sf,) = ax.plot([xs[0], xf[0]], [ys[0], yf[0]], 'b--', lw=2)

ax.set(xlim=[0, screen_resolution[0]], ylim=[0, screen_resolution[1]], xlabel='X [pixels]', ylabel='Y [pixels]')


def init():
    front_ellipse.set_center((xf[0], yf[0]))
    front_ellipse.set_width(bf[0]*2)
    front_ellipse.set_height(af[0]*2)
    front_ellipse.set_angle(thetaf[0])

    rear_ellipse.set_center((xr[0], yr[0]))
    rear_ellipse.set_width(br[0]*2)
    rear_ellipse.set_height(ar[0]*2)
    rear_ellipse.set_angle(thetar[0])

    line.set_data([xr[0], xf[0]], [yr[0], yf[0]])
    # line_rs.set_data([xr[0], xs[0]], [yr[0], ys[0]])
    # line_sf.set_data([xs[0], xf[0]], [ys[0], yf[0]])

    return (front_ellipse, rear_ellipse, line, )


def update(frame):
    front_ellipse.set_center((xf[frame], yf[frame]))
    front_ellipse.set_width(bf[frame]*2)
    front_ellipse.set_height(af[frame]*2)
    front_ellipse.set_angle(thetaf[frame])

    rear_ellipse.set_center((xr[frame], yr[frame]))
    rear_ellipse.set_width(br[frame]*2)
    rear_ellipse.set_height(ar[frame]*2)
    rear_ellipse.set_angle(thetar[frame])

    xs_frame = xr[frame] + 2.5 * ar[frame] * np.cos(iota)
    ys_frame = yr[frame] + 2.5 * ar[frame] * np.sin(iota)

    line.set_data([xr[frame], xf[frame]], [yr[frame], yf[frame]])

    # line_rs.set_data([xr[frame], xs_frame], [yr[frame], ys_frame])
    # line_sf.set_data([xs_frame, xf[frame]], [ys_frame, yf[frame]])    

    ax.set_title(f'Steering angle: {round(np.rad2deg(steering_angle[frame]), 2)}, Frame: {first_frame + frame}, Time: {round(frame/30, 2)} s')
    
    return (front_ellipse, rear_ellipse, line, )



ani = animation.FuncAnimation(fig=fig, func=update, frames=len(xf), interval=60)
# ani.save(filename="yt-crash-005-animation-021125.gif")
HTML(ani.to_jshtml())